# DATA 266 HW2.5 — GPU Assignment I: Precision, Bandwidth, and the Cost of Attention

Single notebook, run top to bottom on the GPU workstation (target: RTX 4090). Not executed here (authored on a Mac with no GPU) -- no cell outputs.

Transcribe real results into `reports/METRICS.md` and `reports/RUN_LOG.txt` after running.


In [1]:
%pip install -q torch numpy pandas matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
SID4 = 9486
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10
print(f"SID4={SID4}, SEED={SEED}, SLICE={SLICE}, HP_ID={HP_ID}, CLS_A={CLS_A}, CLS_B={CLS_B}")


SID4=9486, SEED=9486, SLICE=486, HP_ID=0, CLS_A=6, CLS_B=0


In [3]:
import csv
import json
import math
import random
import subprocess
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_ROOT = Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
for sub in ["system_info", "precision", "bandwidth", "attention", "thermal"]:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA available:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("Device name:", torch.cuda.get_device_name(0))


CUDA available: True
Device name: NVIDIA GeForce RTX 4090


In [4]:
def get_gpu_uuid():
    out = subprocess.run(["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
                          capture_output=True, text=True, check=True)
    return out.stdout.strip().splitlines()[0]

GPU_UUID = get_gpu_uuid() if CUDA_AVAILABLE else None
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None
print("GPU UUID:", GPU_UUID)
print("GPU name:", GPU_NAME)


GPU UUID: GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f
GPU name: NVIDIA GeForce RTX 4090


In [5]:
RAW_LOG_PATH = RESULTS_DIR / "run_log_raw.txt"

def log_event(section, payload):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    line = f"[{stamp}] [UUID {GPU_UUID}] {section} {json.dumps(payload, default=str)}"
    print(line)
    with RAW_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

if not RAW_LOG_PATH.exists():
    RAW_LOG_PATH.write_text("HW2.5 raw measurement log\n")


## Part A — Onboarding and Provenance

Capture `nvidia-smi -q`, measured hardware facts, and vendor-documented specs (kept separate from measured values).

In [6]:
smi_q = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True, check=True)
smi_path = RESULTS_DIR / "system_info" / "nvidia_smi_full_query.txt"
smi_path.write_text(smi_q.stdout)
print("Saved", smi_path)


Saved /app/results/system_info/nvidia_smi_full_query.txt


In [7]:
fields = "uuid,name,driver_version,memory.total,power.limit,power.max_limit,compute_cap"
q = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, check=True)
values = [v.strip() for v in q.stdout.strip().splitlines()[0].split(",")]
measured_hardware_info = dict(zip(fields.split(","), values))
measured_hardware_info["captured_at_utc"] = datetime.now(timezone.utc).isoformat()
measured_hardware_info["torch_version"] = torch.__version__
measured_hardware_info["torch_cuda_version"] = torch.version.cuda

measured_path = RESULTS_DIR / "system_info" / "measured_hardware_info.json"
measured_path.write_text(json.dumps(measured_hardware_info, indent=2))
log_event("SYSTEM_INFO", measured_hardware_info)
measured_hardware_info


[2026-09-18 22:18:27 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] SYSTEM_INFO {"uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "name": "NVIDIA GeForce RTX 4090", "driver_version": "610.60", "memory.total": "24564", "power.limit": "450.00", "power.max_limit": "450.00", "compute_cap": "8.9", "captured_at_utc": "2026-09-18T22:18:27.256780+00:00", "torch_version": "2.1.2", "torch_cuda_version": "12.1"}


{'uuid': 'GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f',
 'name': 'NVIDIA GeForce RTX 4090',
 'driver_version': '610.60',
 'memory.total': '24564',
 'power.limit': '450.00',
 'power.max_limit': '450.00',
 'compute_cap': '8.9',
 'captured_at_utc': '2026-09-18T22:18:27.256780+00:00',
 'torch_version': '2.1.2',
 'torch_cuda_version': '12.1'}

In [8]:
# Vendor-documented (not measured); edit to match measured_hardware_info['name'].
VENDOR_SPECS = {
    "gpu_name": "NVIDIA GeForce RTX 4090",
    "architecture": "Ada Lovelace (AD102)",
    "memory_type": "GDDR6X",
    "memory_bandwidth_gbps_spec": 1008.0,
    "vram_capacity_gb_spec": 24,
    "power_limit_w_spec": 450,
    "tensor_core_generation": "4th generation (Ada)",
    "reduced_precisions_supported": ["FP16", "BF16", "TF32", "INT8", "INT4", "FP8"],
    "theoretical_peak_tflops": {"fp32": 82.6, "tf32": 82.6, "fp16": 330.3, "bf16": 165.2},
    "vendor_documentation_citation": "https://www.nvidia.com/en-us/geforce/graphics-cards/40-series/rtx-4090/",
}

vendor_path = RESULTS_DIR / "system_info" / "vendor_specs.json"
vendor_path.write_text(json.dumps(VENDOR_SPECS, indent=2))
print("GPU UUID for this run:", GPU_UUID)
VENDOR_SPECS


GPU UUID for this run: GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f


{'gpu_name': 'NVIDIA GeForce RTX 4090',
 'architecture': 'Ada Lovelace (AD102)',
 'memory_type': 'GDDR6X',
 'memory_bandwidth_gbps_spec': 1008.0,
 'vram_capacity_gb_spec': 24,
 'power_limit_w_spec': 450,
 'tensor_core_generation': '4th generation (Ada)',
 'reduced_precisions_supported': ['FP16',
  'BF16',
  'TF32',
  'INT8',
  'INT4',
  'FP8'],
 'theoretical_peak_tflops': {'fp32': 82.6,
  'tf32': 82.6,
  'fp16': 330.3,
  'bf16': 165.2},
 'vendor_documentation_citation': 'https://www.nvidia.com/en-us/geforce/graphics-cards/40-series/rtx-4090/'}

## Part B — Precision and Achieved Throughput

Dense matmul at N = 1024, 4096, 8192, 16384 for FP32/TF32/FP16/BF16: warm up, sync before/after, achieved TFLOPS and % of theoretical peak.

In [9]:
MATRIX_SIZES = [1024, 4096, 8192, 16384]
PRECISIONS = ["fp32", "tf32", "fp16", "bf16"]
REPS_BY_SIZE = {1024: 50, 4096: 30, 8192: 15, 16384: 8}
WARMUP_BY_SIZE = {1024: 10, 4096: 8, 8192: 5, 16384: 3}


def dtype_for(precision):
    return {"fp32": torch.float32, "tf32": torch.float32,
            "fp16": torch.float16, "bf16": torch.bfloat16}[precision]


def benchmark_matmul(n, precision):
    torch.backends.cuda.matmul.allow_tf32 = (precision == "tf32")
    dtype = dtype_for(precision)
    reps, warmup = REPS_BY_SIZE[n], WARMUP_BY_SIZE[n]
    row = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "gpu_name": GPU_NAME, "precision": precision, "matrix_n": n,
        "warmup_iters": warmup, "repetitions": reps, "status": "success", "error_message": "",
    }
    try:
        torch.cuda.empty_cache()
        a = torch.randn(n, n, device="cuda", dtype=dtype)
        b = torch.randn(n, n, device="cuda", dtype=dtype)
        for _ in range(warmup):
            _ = a @ b
        torch.cuda.synchronize()

        latencies_ms = []
        for _ in range(reps):
            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = a @ b
            torch.cuda.synchronize()
            latencies_ms.append((time.perf_counter() - start) * 1000.0)

        mean_ms = sum(latencies_ms) / len(latencies_ms)
        flops = 2 * (n ** 3)
        tflops = flops / (mean_ms / 1000.0) / 1e12
        peak = VENDOR_SPECS["theoretical_peak_tflops"][precision]
        row.update({
            "mean_latency_ms": mean_ms,
            "achieved_tflops": tflops,
            "theoretical_peak_tflops": peak,
            "pct_of_theoretical_peak": tflops / peak * 100.0 if peak else float("nan"),
        })
        del a, b
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        row.update({"status": "oom", "error_message": f"{type(exc).__name__}: {exc}"})
    finally:
        torch.cuda.empty_cache()
    return row


precision_rows = [benchmark_matmul(n, p) for n in MATRIX_SIZES for p in PRECISIONS]
for row in precision_rows:
    log_event("MATMUL", row)
precision_df = pd.DataFrame(precision_rows)
precision_df.to_csv(RESULTS_DIR / "precision" / "precision_benchmark_results.csv", index=False)
precision_df


[2026-09-18 22:18:33 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] MATMUL {"timestamp_utc": "2026-09-18T22:18:27.278860+00:00", "gpu_uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "gpu_name": "NVIDIA GeForce RTX 4090", "precision": "fp32", "matrix_n": 1024, "warmup_iters": 10, "repetitions": 50, "status": "success", "error_message": "", "mean_latency_ms": 0.07830265999928088, "achieved_tflops": 27.425423964137643, "theoretical_peak_tflops": 82.6, "pct_of_theoretical_peak": 33.20269245052984}
[2026-09-18 22:18:33 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] MATMUL {"timestamp_utc": "2026-09-18T22:18:27.526802+00:00", "gpu_uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "gpu_name": "NVIDIA GeForce RTX 4090", "precision": "tf32", "matrix_n": 1024, "warmup_iters": 10, "repetitions": 50, "status": "success", "error_message": "", "mean_latency_ms": 0.0682130199970743, "achieved_tflops": 31.482019826890923, "theoretical_peak_tflops": 82.6, "pct_of_theoretical_peak": 38.1

,timestamp_utc,gpu_uuid,gpu_name,precision,matrix_n,warmup_iters,repetitions,status,error_message,mean_latency_ms,achieved_tflops,theoretical_peak_tflops,pct_of_theoretical_peak
0,2026-09-18T22:18:27.278860+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,fp32,1024,10,50,success,,0.078303,27.425424,82.6,33.202692
1,2026-09-18T22:18:27.526802+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,tf32,1024,10,50,success,,0.068213,31.482020,82.6,38.113825
2,2026-09-18T22:18:27.576457+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,fp16,1024,10,50,success,,0.056379,38.089985,330.3,11.531936
3,2026-09-18T22:18:27.704102+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,bf16,1024,10,50,success,,0.055469,38.715247,165.2,23.435380
4,2026-09-18T22:18:27.796678+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,fp32,4096,8,30,success,,2.868096,47.919933,82.6,58.014447
5,2026-09-18T22:18:27.913981+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,tf32,4096,8,30,success,,1.814412,75.748474,82.6,91.705175
6,2026-09-18T22:18:27.991433+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,fp16,4096,8,30,success,,0.841372,163.350924,330.3,49.455321
7,2026-09-18T22:18:28.027725+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,bf16,4096,8,30,success,,0.839512,163.712989,165.2,99.099872
8,2026-09-18T22:18:28.063523+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,fp32,8192,5,15,success,,21.283380,51.660574,82.6,62.543068
9,2026-09-18T22:18:28.504211+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,NVIDIA GeForce RTX 4090,tf32,8192,5,15,success,,12.716887,86.460750,82.6,104.674032


In [10]:
def attempt_fp8_probe(n=4096, reps=20, warmup=5):
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "precision_attempted": "fp8_e4m3", "api_attempted": "torch._scaled_mm",
    }
    try:
        if not hasattr(torch, "float8_e4m3fn") or not hasattr(torch, "_scaled_mm"):
            raise RuntimeError("torch.float8_e4m3fn or torch._scaled_mm not available in this PyTorch build.")
        a = torch.randn(n, n, device="cuda", dtype=torch.float32).to(torch.float8_e4m3fn)
        b = torch.randn(n, n, device="cuda", dtype=torch.float32).to(torch.float8_e4m3fn)
        scale = torch.tensor(1.0, device="cuda")
        for _ in range(warmup):
            _ = torch._scaled_mm(a, b.t(), scale_a=scale, scale_b=scale, out_dtype=torch.bfloat16)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(reps):
            _ = torch._scaled_mm(a, b.t(), scale_a=scale, scale_b=scale, out_dtype=torch.bfloat16)
        torch.cuda.synchronize()
        mean_ms = (time.perf_counter() - start) * 1000.0 / reps
        tflops = 2 * (n ** 3) / (mean_ms / 1000.0) / 1e12
        record.update({"status": "available", "mean_latency_ms": mean_ms, "achieved_tflops": tflops})
    except Exception as exc:
        record.update({"status": "unavailable", "error_type": type(exc).__name__, "error_message": str(exc)})
    return record


fp8_result = attempt_fp8_probe()
(RESULTS_DIR / "precision" / "fp8_availability_probe.json").write_text(json.dumps(fp8_result, indent=2))
log_event("FP8", fp8_result)
fp8_result


[2026-09-18 22:18:33 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] FP8 {"timestamp_utc": "2026-09-18T22:18:33.462403+00:00", "gpu_uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "precision_attempted": "fp8_e4m3", "api_attempted": "torch._scaled_mm", "status": "unavailable", "error_type": "RuntimeError", "error_message": "CUDA error: CUBLAS_STATUS_NOT_SUPPORTED when calling `cublasLtMatmulAlgoGetHeuristic( ltHandle, computeDesc.descriptor(), Adesc.descriptor(), Bdesc.descriptor(), Cdesc.descriptor(), Ddesc.descriptor(), preference.descriptor(), 1, &heuristicResult, &returnedResult)`"}


{'timestamp_utc': '2026-09-18T22:18:33.462403+00:00',
 'gpu_uuid': 'GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f',
 'precision_attempted': 'fp8_e4m3',
 'api_attempted': 'torch._scaled_mm',
 'status': 'unavailable',
 'error_type': 'RuntimeError',
 'error_message': 'CUDA error: CUBLAS_STATUS_NOT_SUPPORTED when calling `cublasLtMatmulAlgoGetHeuristic( ltHandle, computeDesc.descriptor(), Adesc.descriptor(), Bdesc.descriptor(), Cdesc.descriptor(), Ddesc.descriptor(), preference.descriptor(), 1, &heuristicResult, &returnedResult)`'}

In [11]:
fig, ax = plt.subplots(figsize=(8, 5.5))
successful = precision_df[precision_df.status == "success"]
for precision, group in successful.groupby("precision"):
    group = group.sort_values("matrix_n")
    ax.plot(group.matrix_n, group.achieved_tflops, marker="o", label=precision.upper())
ax.set_xscale("log", base=2)
ax.set_xlabel("Matrix size N (N x N)")
ax.set_ylabel("Achieved TFLOPS")
ax.set_title("Part B — Achieved TFLOPS vs. Matrix Size by Precision")
ax.legend(title="Precision")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "precision_tflops_vs_matrix_size.png", dpi=200)
plt.show()


**Plateau discussion (fill in from `precision_df` after running):** matrix size where each precision plateaus, and why smaller matrices fall short (launch overhead, underfilled SMs, or memory-bound per Part C's roofline).

## Part C — Bandwidth-Bound vs. Compute-Bound

Memory-bound elementwise add and compute-bound square matmul, compared against the roofline ridge point. The add's tensor size shrinks and retries on OOM.

In [12]:
def roofline_ridge_point():
    peak_flops = VENDOR_SPECS["theoretical_peak_tflops"]["fp32"] * 1e12
    peak_bytes_per_s = VENDOR_SPECS["memory_bandwidth_gbps_spec"] * 1e9
    return peak_flops / peak_bytes_per_s


def classify(intensity, ridge_point):
    return "compute_bound" if intensity >= ridge_point else "bandwidth_bound"


def benchmark_elementwise_add(target_n=400_000_000, min_n=12_500_000, reps=20, warmup=5):
    n = target_n
    while n >= min_n:
        try:
            torch.cuda.empty_cache()
            a = torch.randn(n, device="cuda", dtype=torch.float32)
            b = torch.randn(n, device="cuda", dtype=torch.float32)
            for _ in range(warmup):
                c = a + b
            torch.cuda.synchronize()

            latencies_ms = []
            for _ in range(reps):
                torch.cuda.synchronize()
                start = time.perf_counter()
                c = a + b
                torch.cuda.synchronize()
                latencies_ms.append((time.perf_counter() - start) * 1000.0)

            mean_ms = sum(latencies_ms) / len(latencies_ms)
            bytes_moved = 3 * n * 4
            flops = n
            gbps = (bytes_moved / (mean_ms / 1000.0)) / 1e9
            pct = (gbps / VENDOR_SPECS["memory_bandwidth_gbps_spec"]) * 100.0
            intensity = flops / bytes_moved
            ridge = roofline_ridge_point()
            del a, b, c
            torch.cuda.empty_cache()
            return {
                "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
                "operation": "elementwise_add", "category": "memory_bound", "n": n,
                "status": "success", "mean_latency_ms": mean_ms, "achieved_gbps": gbps,
                "pct_of_spec_bandwidth": pct, "arithmetic_intensity_flops_per_byte": intensity,
                "roofline_ridge_point_flops_per_byte": ridge, "roofline_classification": classify(intensity, ridge),
            }
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            torch.cuda.empty_cache()
            n //= 2
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "operation": "elementwise_add", "category": "memory_bound", "n": None,
        "status": "oom", "note": f"Did not fit even at the minimum size ({min_n} elements).",
    }


def benchmark_compute_bound_matmul(n=8192, reps=15, warmup=5):
    torch.backends.cuda.matmul.allow_tf32 = False
    a = torch.randn(n, n, device="cuda", dtype=torch.float32)
    b = torch.randn(n, n, device="cuda", dtype=torch.float32)
    for _ in range(warmup):
        _ = a @ b
    torch.cuda.synchronize()

    latencies_ms = []
    for _ in range(reps):
        torch.cuda.synchronize()
        start = time.perf_counter()
        _ = a @ b
        torch.cuda.synchronize()
        latencies_ms.append((time.perf_counter() - start) * 1000.0)

    mean_ms = sum(latencies_ms) / len(latencies_ms)
    flops = 2 * (n ** 3)
    bytes_moved = 3 * (n ** 2) * 4
    gbps = (bytes_moved / (mean_ms / 1000.0)) / 1e9
    intensity = flops / bytes_moved
    ridge = roofline_ridge_point()
    del a, b
    torch.cuda.empty_cache()
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "operation": "square_matmul", "category": "compute_bound", "n": n,
        "status": "success", "mean_latency_ms": mean_ms, "achieved_gbps": gbps,
        "arithmetic_intensity_flops_per_byte": intensity,
        "roofline_ridge_point_flops_per_byte": ridge, "roofline_classification": classify(intensity, ridge),
    }


bandwidth_rows = [benchmark_elementwise_add(), benchmark_compute_bound_matmul()]
for row in bandwidth_rows:
    log_event("BANDWIDTH", row)
bandwidth_df = pd.DataFrame(bandwidth_rows)
bandwidth_df.to_csv(RESULTS_DIR / "bandwidth" / "bandwidth_benchmark_results.csv", index=False)
bandwidth_df


[2026-09-18 22:18:34 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] BANDWIDTH {"timestamp_utc": "2026-09-18T22:18:34.002203+00:00", "gpu_uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "operation": "elementwise_add", "category": "memory_bound", "n": 400000000, "status": "success", "mean_latency_ms": 5.347057149998591, "achieved_gbps": 897.6900499373314, "pct_of_spec_bandwidth": 89.05655257314797, "arithmetic_intensity_flops_per_byte": 0.08333333333333333, "roofline_ridge_point_flops_per_byte": 81.94444444444444, "roofline_classification": "bandwidth_bound"}
[2026-09-18 22:18:34 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] BANDWIDTH {"timestamp_utc": "2026-09-18T22:18:34.434345+00:00", "gpu_uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "operation": "square_matmul", "category": "compute_bound", "n": 8192, "status": "success", "mean_latency_ms": 21.11481413333725, "achieved_gbps": 38.139401223927294, "arithmetic_intensity_flops_per_byte": 1365.3333333333333, "rooflin

,timestamp_utc,gpu_uuid,operation,category,n,status,mean_latency_ms,achieved_gbps,pct_of_spec_bandwidth,arithmetic_intensity_flops_per_byte,roofline_ridge_point_flops_per_byte,roofline_classification
0,2026-09-18T22:18:34.002203+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,elementwise_add,memory_bound,400000000,success,5.347057,897.690050,89.056553,0.083333,81.944444,bandwidth_bound
1,2026-09-18T22:18:34.434345+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,square_matmul,compute_bound,8192,success,21.114814,38.139401,NaN,1365.333333,81.944444,compute_bound


## Part D — Cost of Attention

Naive attention (materializes the full `[S, S]` matrix) vs. fused `scaled_dot_product_attention`. B=1, 1 head, D=64, bfloat16, `torch.inference_mode()`. OOM is caught per length; the boundary between success and failure is bisected, never claimed to single-token precision.

In [13]:
BATCH_SIZE, NUM_HEADS, HEAD_DIM = 1, 1, 64
ATTENTION_DTYPE = torch.bfloat16
SEQ_LENGTHS = [512, 1024, 2048, 4096, 8192, 16384]
ATTENTION_REPS, ATTENTION_WARMUP = 10, 3
MAX_BOUNDARY_PROBES = 6
MAX_EXTENSION_STEPS = 12
EXTENSION_TIME_BUDGET_S = 120


def naive_attention(q, k, v):
    d_k = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    return weights @ v


def fused_attention(q, k, v):
    try:
        from torch.nn.attention import SDPBackend, sdpa_kernel
        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):
            return F.scaled_dot_product_attention(q, k, v)
    except ImportError:
        with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=True):
            return F.scaled_dot_product_attention(q, k, v)


def run_attention_once(seq_len, implementation, probe_type="grid"):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    row = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "implementation": implementation, "dtype": "bfloat16", "batch_size": BATCH_SIZE,
        "num_heads": NUM_HEADS, "head_dim": HEAD_DIM, "seq_len": seq_len,
        "status": "success", "error_message": "", "probe_type": probe_type,
    }
    fn = naive_attention if implementation == "naive" else fused_attention
    try:
        shape = (BATCH_SIZE, NUM_HEADS, seq_len, HEAD_DIM)
        q = torch.randn(shape, device="cuda", dtype=ATTENTION_DTYPE)
        k = torch.randn(shape, device="cuda", dtype=ATTENTION_DTYPE)
        v = torch.randn(shape, device="cuda", dtype=ATTENTION_DTYPE)
        with torch.inference_mode():
            for _ in range(ATTENTION_WARMUP):
                _ = fn(q, k, v)
            torch.cuda.synchronize()
            latencies_ms = []
            for _ in range(ATTENTION_REPS):
                torch.cuda.synchronize()
                start = time.perf_counter()
                _ = fn(q, k, v)
                torch.cuda.synchronize()
                latencies_ms.append((time.perf_counter() - start) * 1000.0)
        row["mean_latency_ms"] = sum(latencies_ms) / len(latencies_ms)
        row["peak_memory_mb"] = torch.cuda.max_memory_allocated() / (1024 ** 2)
        del q, k, v
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        row["status"] = "oom"
        row["error_message"] = f"{type(exc).__name__}: {exc}"
        row["peak_memory_mb"] = torch.cuda.max_memory_allocated() / (1024 ** 2)
    finally:
        torch.cuda.empty_cache()
    return row


def find_oom_boundary(implementation, grid_rows):
    extra_rows = []
    successes = sorted(r["seq_len"] for r in grid_rows if r["status"] == "success")
    failures = sorted(r["seq_len"] for r in grid_rows if r["status"] == "oom")

    candidate = (max(successes) * 2) if successes else (2 * SEQ_LENGTHS[-1])
    for _ in range(MAX_EXTENSION_STEPS):
        if failures:
            break
        start = time.perf_counter()
        probe = run_attention_once(candidate, implementation, probe_type="extension")
        elapsed_s = time.perf_counter() - start
        extra_rows.append(probe)
        if probe["status"] == "success":
            successes.append(candidate)
            if elapsed_s > EXTENSION_TIME_BUDGET_S:
                break
            candidate *= 2
        else:
            failures.append(candidate)

    if not failures:
        return extra_rows

    low, high = max(successes), min(failures)
    for _ in range(MAX_BOUNDARY_PROBES):
        if high - low <= 64:
            break
        mid = ((low + high) // 2 // 64) * 64
        if mid <= low or mid >= high:
            break
        probe = run_attention_once(mid, implementation, probe_type="boundary_refinement")
        extra_rows.append(probe)
        if probe["status"] == "success":
            low = mid
        else:
            high = mid
    return extra_rows


attention_rows = []
for implementation in ("naive", "fused"):
    grid_rows = [run_attention_once(s, implementation) for s in SEQ_LENGTHS]
    for row in grid_rows:
        log_event("ATTENTION", row)
    attention_rows.extend(grid_rows)

    boundary_rows = find_oom_boundary(implementation, grid_rows)
    for row in boundary_rows:
        log_event("ATTENTION_REFINE", row)
    attention_rows.extend(boundary_rows)

attention_df = pd.DataFrame(attention_rows)
attention_df.to_csv(RESULTS_DIR / "attention" / "attention_benchmark_results.csv", index=False)
attention_df


[2026-09-18 22:18:34 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] ATTENTION {"timestamp_utc": "2026-09-18T22:18:34.460508+00:00", "gpu_uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "implementation": "naive", "dtype": "bfloat16", "batch_size": 1, "num_heads": 1, "head_dim": 64, "seq_len": 512, "status": "success", "error_message": "", "probe_type": "grid", "mean_latency_ms": 0.059922100012954616, "peak_memory_mb": 9.375}
[2026-09-18 22:18:34 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] ATTENTION {"timestamp_utc": "2026-09-18T22:18:34.511197+00:00", "gpu_uuid": "GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f", "implementation": "naive", "dtype": "bfloat16", "batch_size": 1, "num_heads": 1, "head_dim": 64, "seq_len": 1024, "status": "success", "error_message": "", "probe_type": "grid", "mean_latency_ms": 0.06018560002303275, "peak_memory_mb": 12.625}
[2026-09-18 22:18:34 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] ATTENTION {"timestamp_utc": "2026-09-18T22:18:34.51

,timestamp_utc,gpu_uuid,implementation,dtype,batch_size,num_heads,head_dim,seq_len,status,error_message,probe_type,mean_latency_ms,peak_memory_mb
0,2026-09-18T22:18:34.460508+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,512,success,,grid,0.059922,9.375
1,2026-09-18T22:18:34.511197+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,1024,success,,grid,0.060186,12.625
2,2026-09-18T22:18:34.514083+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,2048,success,,grid,0.081368,25.125
3,2026-09-18T22:18:34.516667+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,4096,success,,grid,0.154721,74.125
4,2026-09-18T22:18:34.521315+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,8192,success,,grid,0.967236,268.125
5,2026-09-18T22:18:34.541367+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,16384,success,,grid,3.643435,1040.125
6,2026-09-18T22:18:34.634856+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,32768,success,,extension,15.068345,4120.125
7,2026-09-18T22:18:34.931938+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,65536,success,,extension,70.337084,16424.125
8,2026-09-18T22:18:36.243249+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,131072,success,,extension,6327.749681,65608.125
9,2026-09-18T22:20:10.311281+00:00,GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f,naive,bfloat16,1,1,64,262144,oom,OutOfMemoryError: CUDA out of memory. Tried to...,extension,NaN,104.125


In [14]:
def oom_boundary(df, implementation):
    subset = df[df.implementation == implementation]
    successes = subset.loc[subset.status == "success", "seq_len"]
    failures = subset.loc[subset.status == "oom", "seq_len"]
    return {
        "implementation": implementation,
        "largest_tested_success": int(successes.max()) if len(successes) else None,
        "smallest_tested_failure": int(failures.min()) if len(failures) else None,
    }


naive_boundary = oom_boundary(attention_df, "naive")
fused_boundary = oom_boundary(attention_df, "fused")

naive_success = attention_df[
    (attention_df.implementation == "naive") & (attention_df.status == "success")
].sort_values("seq_len")
if len(naive_success) >= 3:
    a_coeff, b_coeff, c_coeff = np.polyfit(naive_success.seq_len, naive_success.peak_memory_mb, deg=2)
    memory_fit = {"status": "fit", "quadratic_coefficient_a": float(a_coeff),
                  "linear_coefficient_b": float(b_coeff), "intercept_c": float(c_coeff)}
else:
    memory_fit = {"status": "insufficient_data"}

naive_latency = attention_df[
    (attention_df.implementation == "naive") & (attention_df.status == "success")
].set_index("seq_len")["mean_latency_ms"]
fused_latency = attention_df[
    (attention_df.implementation == "fused") & (attention_df.status == "success")
].set_index("seq_len")["mean_latency_ms"]
common_lengths = sorted(set(naive_latency.index) & set(fused_latency.index))
speedups = [
    {"seq_len": s, "naive_latency_ms": float(naive_latency[s]), "fused_latency_ms": float(fused_latency[s]),
     "speedup_naive_over_fused": float(naive_latency[s] / fused_latency[s])}
    for s in common_lengths
]

attention_summary = {
    "gpu_uuid": GPU_UUID,
    "naive_oom_boundary": naive_boundary,
    "fused_oom_boundary": fused_boundary,
    "memory_quadratic_fit_naive": memory_fit,
    "speedups_fused_over_naive": speedups,
}
(RESULTS_DIR / "attention" / "attention_summary.json").write_text(json.dumps(attention_summary, indent=2))
attention_summary


{'gpu_uuid': 'GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f',
 'naive_oom_boundary': {'implementation': 'naive',
  'largest_tested_success': 182272,
  'smallest_tested_failure': 184320},
 'fused_oom_boundary': {'implementation': 'fused',
  'largest_tested_success': 2097152,
  'smallest_tested_failure': None},
 'memory_quadratic_fit_naive': {'status': 'fit',
  'quadratic_coefficient_a': 3.814697265625002e-06,
  'linear_coefficient_b': 0.00048828124999968233,
  'intercept_c': 8.125000000017549},
 'speedups_fused_over_naive': [{'seq_len': 512,
   'naive_latency_ms': 0.059922100012954616,
   'fused_latency_ms': 0.1796732999764572,
   'speedup_naive_over_fused': 0.3335058688230598},
  {'seq_len': 1024,
   'naive_latency_ms': 0.06018560002303275,
   'fused_latency_ms': 0.2135437999640999,
   'speedup_naive_over_fused': 0.28184194546107594},
  {'seq_len': 2048,
   'naive_latency_ms': 0.08136820001709566,
   'fused_latency_ms': 0.39155259999006375,
   'speedup_naive_over_fused': 0.20780911688279047

In [15]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for implementation, marker in (("naive", "o"), ("fused", "s")):
    d = attention_df[
        (attention_df.implementation == implementation) & (attention_df.status == "success")
    ].sort_values("seq_len")
    if len(d):
        ax.plot(d.seq_len, d.peak_memory_mb, marker=marker, label=f"{implementation} (success)")
    failed = attention_df[
        (attention_df.implementation == implementation) & (attention_df.status == "oom")
    ].seq_len
    for x in failed:
        ax.axvline(x, color="red", linestyle=":", alpha=0.3)

if memory_fit.get("status") == "fit":
    xs_fit = np.linspace(naive_success.seq_len.min(), naive_success.seq_len.max(), 200)
    ys_fit = memory_fit["quadratic_coefficient_a"] * xs_fit ** 2 + memory_fit["linear_coefficient_b"] * xs_fit + memory_fit["intercept_c"]
    ax.plot(xs_fit, ys_fit, linestyle="--", color="black", alpha=0.6, label="naive quadratic fit")

ax.set_xlabel("Sequence length")
ax.set_ylabel("Peak allocated GPU memory (MB)")
ax.set_title("Part D — Peak Memory vs. Sequence Length (naive vs. fused attention)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "attention_peak_memory_vs_seqlen.png", dpi=200)
plt.show()


**What the fused kernel avoids:** it never materializes the full `[S, S]` score/probability matrix. It processes Q/K/V in tiles with an online softmax, keeping only small per-tile buffers resident, so peak memory scales close to `O(S)` instead of `O(S^2)` — which is why it keeps succeeding past the naive implementation's OOM boundary.

## Part E — Sustained Load and Thermal Behaviour

~20-minute sustained matmul load, sampling GPU clocks/temperature/power/utilization every 5 seconds; CSV flushed every row. This cell blocks for the full duration.

In [16]:
def sample_gpu():
    fields = "uuid,clocks.sm,clocks.mem,temperature.gpu,power.draw,utilization.gpu"
    out = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True, check=True)
    uuid, sm, mem, temp, power, util = [v.strip() for v in out.stdout.strip().splitlines()[0].split(",")]
    return {"gpu_uuid": uuid, "sm_clock_mhz": float(sm), "memory_clock_mhz": float(mem),
            "temperature_c": float(temp), "power_draw_w": float(power), "utilization_pct": float(util)}


CSV_FIELDS = [
    "elapsed_s", "timestamp_utc", "gpu_uuid", "sm_clock_mhz", "memory_clock_mhz",
    "temperature_c", "power_draw_w", "utilization_pct", "cumulative_matmuls",
    "interval_throughput_matmuls_per_s",
]


def run_sustained_load(duration_s=1200, interval_s=5.0, matmul_n=8192, output_csv=None):
    output_csv = output_csv or (RESULTS_DIR / "thermal" / "thermal_log.csv")
    a = torch.randn(matmul_n, matmul_n, device="cuda", dtype=torch.float16)
    b = torch.randn(matmul_n, matmul_n, device="cuda", dtype=torch.float16)
    counter = {"n": 0}
    stop_event = threading.Event()

    def sampler():
        start_time = time.time()
        last_count, last_sample_time = 0, start_time
        next_sample_at = start_time + interval_s
        with output_csv.open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
            writer.writeheader()
            f.flush()
            while not stop_event.is_set():
                now = time.time()
                if now < next_sample_at:
                    time.sleep(min(0.25, next_sample_at - now))
                    continue
                try:
                    sample = sample_gpu()
                except subprocess.CalledProcessError:
                    sample = {"gpu_uuid": "UNKNOWN", "sm_clock_mhz": float("nan"),
                              "memory_clock_mhz": float("nan"), "temperature_c": float("nan"),
                              "power_draw_w": float("nan"), "utilization_pct": float("nan")}
                now = time.time()
                current_count = counter["n"]
                dt = now - last_sample_time
                throughput = (current_count - last_count) / dt if dt > 0 else float("nan")
                row = {
                    "elapsed_s": round(now - start_time, 3),
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    **sample,
                    "cumulative_matmuls": current_count,
                    "interval_throughput_matmuls_per_s": throughput,
                }
                writer.writerow(row)
                f.flush()
                last_count, last_sample_time = current_count, now
                next_sample_at += interval_s

    sampler_thread = threading.Thread(target=sampler, daemon=True)
    sampler_thread.start()
    print(f"Starting sustained load: duration={duration_s}s, interval={interval_s}s, matmul_n={matmul_n}.")
    end_time = time.time() + duration_s
    try:
        while time.time() < end_time:
            _ = a @ b
            counter["n"] += 1
    except KeyboardInterrupt:
        print("Interrupted; saving partial log.")
    finally:
        torch.cuda.synchronize()
        stop_event.set()
        sampler_thread.join(timeout=interval_s * 2)
    print(f"Sustained load complete. Log saved to {output_csv}")
    return output_csv


thermal_csv = run_sustained_load(duration_s=1200, interval_s=5.0)


Starting sustained load: duration=1200s, interval=5.0s, matmul_n=8192.
Sustained load complete. Log saved to /app/results/thermal/thermal_log.csv


In [17]:
def analyze_thermal_log(csv_path):
    thermal_df = pd.read_csv(csv_path)
    if thermal_df.empty:
        return {"status": "no_data"}

    total_duration = thermal_df.elapsed_s.max()
    first_30s = thermal_df[thermal_df.elapsed_s <= 30]
    last_5min = thermal_df[thermal_df.elapsed_s >= total_duration - 300]

    peak_clock = first_30s.sm_clock_mhz.max()
    threshold = 0.97 * peak_clock if pd.notna(peak_clock) else float("nan")
    throttle_onset_s = None
    throttle_onset_temperature_c = None
    throttle_onset_power_w = None
    consecutive_low = 0
    for _, row in thermal_df[thermal_df.elapsed_s > 30].iterrows():
        if pd.notna(row.sm_clock_mhz) and row.sm_clock_mhz <= threshold:
            consecutive_low += 1
            if consecutive_low >= 2 and throttle_onset_s is None:
                throttle_onset_s = row.elapsed_s
                throttle_onset_temperature_c = row.temperature_c
                throttle_onset_power_w = row.power_draw_w
        else:
            consecutive_low = 0

    peak_throughput = first_30s.interval_throughput_matmuls_per_s.max()
    steady_throughput = last_5min.interval_throughput_matmuls_per_s.mean()
    steady_pct_of_peak = (
        steady_throughput / peak_throughput * 100.0
        if pd.notna(peak_throughput) and peak_throughput else float("nan")
    )

    return {
        "total_logged_duration_s": float(total_duration),
        "num_samples": len(thermal_df),
        "peak_sm_clock_mhz_first_30s": float(peak_clock) if pd.notna(peak_clock) else None,
        "max_temperature_c_observed": float(thermal_df.temperature_c.max()),
        "max_power_draw_w_observed": float(thermal_df.power_draw_w.max()),
        "throttling_detected": throttle_onset_s is not None,
        "throttle_onset_time_s": float(throttle_onset_s) if throttle_onset_s is not None else "none",
        "temperature_c_at_throttle_onset": float(throttle_onset_temperature_c) if throttle_onset_temperature_c is not None else "none",
        "power_draw_w_at_throttle_onset": float(throttle_onset_power_w) if throttle_onset_power_w is not None else "none",
        "peak_throughput_matmuls_per_s_first_30s": float(peak_throughput) if pd.notna(peak_throughput) else None,
        "steady_state_throughput_matmuls_per_s_last_5min": float(steady_throughput) if pd.notna(steady_throughput) else None,
        "steady_state_pct_of_peak_throughput": float(steady_pct_of_peak) if pd.notna(steady_pct_of_peak) else None,
    }


thermal_analysis = analyze_thermal_log(thermal_csv)
(RESULTS_DIR / "thermal" / "thermal_analysis.json").write_text(json.dumps(thermal_analysis, indent=2))
log_event("THERMAL", {"path": str(thermal_csv), **thermal_analysis})
thermal_analysis


[2026-09-18 22:55:36 UTC] [UUID GPU-00b80185-a51c-ac5b-84b1-c291bcf5025f] THERMAL {"path": "/app/results/thermal/thermal_log.csv", "total_logged_duration_s": 1207.408, "num_samples": 240, "peak_sm_clock_mhz_first_30s": 2400.0, "max_temperature_c_observed": 76.0, "max_power_draw_w_observed": 451.49, "throttling_detected": false, "throttle_onset_time_s": "none", "temperature_c_at_throttle_onset": "none", "power_draw_w_at_throttle_onset": "none", "peak_throughput_matmuls_per_s_first_30s": 339.8452535476421, "steady_state_throughput_matmuls_per_s_last_5min": 136.56635147226854, "steady_state_pct_of_peak_throughput": 40.18486356559446}


{'total_logged_duration_s': 1207.408,
 'num_samples': 240,
 'peak_sm_clock_mhz_first_30s': 2400.0,
 'max_temperature_c_observed': 76.0,
 'max_power_draw_w_observed': 451.49,
 'throttling_detected': False,
 'throttle_onset_time_s': 'none',
 'temperature_c_at_throttle_onset': 'none',
 'power_draw_w_at_throttle_onset': 'none',
 'peak_throughput_matmuls_per_s_first_30s': 339.8452535476421,
 'steady_state_throughput_matmuls_per_s_last_5min': 136.56635147226854,
 'steady_state_pct_of_peak_throughput': 40.18486356559446}

In [18]:
thermal_df = pd.read_csv(thermal_csv)

fig, ax1 = plt.subplots(figsize=(9, 5.5))
ax1.plot(thermal_df.elapsed_s, thermal_df.sm_clock_mhz, color="tab:blue", label="SM clock (MHz)")
ax1.plot(thermal_df.elapsed_s, thermal_df.memory_clock_mhz, color="tab:cyan", label="Memory clock (MHz)")
ax1.set_xlabel("Elapsed time (s)")
ax1.set_ylabel("Clock (MHz)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(thermal_df.elapsed_s, thermal_df.temperature_c, color="tab:red", label="Temperature (C)")
ax2.set_ylabel("Temperature (C)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower center")
ax1.set_title("Part E — Clock and Temperature vs. Time (sustained load)")
ax1.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "thermal_clock_temperature_vs_time.png", dpi=200)
plt.show()
